## Project 1 - Fixed Implementation

### Eliminating Child Care Deserts in New York State through Optimization

#### Question 1: Idealistic Situation
#### Question 2: Realistic Situation




## Data Preprocessing

In [ ]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict


In [ ]:
# Activate Gurobi license
# Gurobi reads its license from ~/gurobi.lic (not included in this repo).
# The license will be automatically activated when Gurobi is used.

# Create and store the Gurobi environment to use with all models
# This environment will be used by both optimization models
env = gp.Env()
env.start()
print("✅ Gurobi license activated")
print(f"Gurobi version: {gp.gurobi.version()}")


Academic license - for non-commercial use only - expires 2026-11-04
✅ Gurobi license activated
Gurobi version: (12, 0, 3)


In [ ]:
# 1) Read data files
avg_individual_income = pd.read_csv('ChildCareDeserts_Data/avg_individual_income.csv')
child_care_regulated = pd.read_csv('ChildCareDeserts_Data/child_care_regulated.csv')
employment_rate = pd.read_csv('ChildCareDeserts_Data/employment_rate.csv')
population = pd.read_csv('ChildCareDeserts_Data/population.csv')
potential_locations = pd.read_csv('ChildCareDeserts_Data/potential_locations.csv')

print("Data files loaded successfully")


Data files loaded successfully


In [ ]:
# 2) Standardize ZIP column names to `zipcode`
avg_individual_income = avg_individual_income.rename(columns={'ZIP code': 'zipcode'})
child_care_regulated = child_care_regulated.rename(columns={'zip_code': 'zipcode'})

# 3) Force ZIPs to be strings, zero-pad to 5 digits
dataframes = [avg_individual_income, child_care_regulated, employment_rate, population, potential_locations]
for df in dataframes:
    df['zipcode'] = df['zipcode'].astype(str).str.zfill(5)

# 4) Basic sanity checks
print("Checking for null ZIP codes:")
for name, df in zip(['Income', 'Childcare', 'Employment', 'Population', 'Potential Locations'],
                   [avg_individual_income, child_care_regulated, employment_rate, population, potential_locations]):
    null_count = df['zipcode'].isnull().sum()
    print(f"{name} dataset: {null_count} null ZIP codes")
    assert null_count == 0, f"Found {null_count} null ZIP codes in {name} dataset"


Checking for null ZIP codes:
Income dataset: 0 null ZIP codes
Childcare dataset: 0 null ZIP codes
Employment dataset: 0 null ZIP codes
Population dataset: 0 null ZIP codes
Potential Locations dataset: 0 null ZIP codes


In [ ]:
# 5) Aggregate childcare capacity per ZIP
# Treat infant/toddler/preschool as 0-5 supply, school_age as 5-12 supply
cap_0_5_fac = child_care_regulated['infant_capacity'] + child_care_regulated['toddler_capacity'] + child_care_regulated['preschool_capacity']
cap_5_12_fac = child_care_regulated['school_age_capacity']

childcare_zip = child_care_regulated.groupby('zipcode').agg({
    'infant_capacity': 'sum',
    'toddler_capacity': 'sum',
    'preschool_capacity': 'sum',
    'school_age_capacity': 'sum'
}).reset_index()

childcare_zip['cap_0_5_zip'] = cap_0_5_fac.groupby(child_care_regulated['zipcode']).sum().values
childcare_zip['cap_5_12_zip'] = cap_5_12_fac.groupby(child_care_regulated['zipcode']).sum().values
childcare_zip['cap_total_zip'] = childcare_zip['cap_0_5_zip'] + childcare_zip['cap_5_12_zip']


In [ ]:
# 6) Build Population Buckets
pop_0_5 = population['-5']
pop_5_12 = population['5-9'] + 0.5 * population['10-14']  # assumption: half of 10–14 are 10–12
pop_total_0_12 = pop_0_5 + pop_5_12

population['pop_0_5'] = pop_0_5
population['pop_5_12'] = pop_5_12
population['pop_total_0_12'] = pop_total_0_12


In [ ]:
# 7) Merge to Master ZIP Table
master = population[['zipcode', 'pop_0_5', 'pop_5_12', 'pop_total_0_12']].copy()
master = master.merge(avg_individual_income[['zipcode','average income']], on='zipcode', how='left')
master = master.merge(employment_rate[['zipcode','employment rate']], on='zipcode', how='left')
master = master.merge(childcare_zip[['zipcode','cap_0_5_zip','cap_5_12_zip','cap_total_zip']],
                      on='zipcode', how='left')

# Fill missing capacities with 0
master['cap_total_zip'] = master['cap_total_zip'].fillna(0)
master['cap_0_5_zip'] = master['cap_0_5_zip'].fillna(0)
childcare_zip = childcare_zip.fillna(0)

print(f"Master table created with {len(master)} ZIP codes")


Master table created with 1646 ZIP codes


In [ ]:
# 8) Compute High-Demand Flag
# High-demand if employment_rate >= 0.60 OR avg_income <= 60,000
master['is_high_demand'] = ((master['employment rate'] >= 0.60) |
                            (master['average income'] <= 60000)).astype(int)

print("High-demand ZIPs:", master['is_high_demand'].sum())
print("Normal-demand ZIPs:", (master['is_high_demand'] == 0).sum())


High-demand ZIPs: 561
Normal-demand ZIPs: 1085


In [ ]:
# 9) Build Facility-Level Parameters for Expansion
# Compute total and 0-5 capacity per facility
child_care_regulated['nf'] = (child_care_regulated['infant_capacity'] + 
                               child_care_regulated['toddler_capacity'] + 
                               child_care_regulated['preschool_capacity'] + 
                               child_care_regulated['school_age_capacity'])
child_care_regulated['nf_0_5'] = (child_care_regulated['infant_capacity'] + 
                                  child_care_regulated['toddler_capacity'] + 
                                  child_care_regulated['preschool_capacity'])

# Expansion limits:
# - Idealistic: min(500, 120% of current capacity) = min(500, 0.2 * nf)
# - Realistic: min(500, 20% of current capacity) = min(500, 0.20 * nf)
# Note: Both use 0.2 * nf, but idealistic allows up to 120% total (expansion = 20% of current)
child_care_regulated['idealistic_cap'] = child_care_regulated[['nf']].apply(
    lambda x: min(500, 0.20 * x['nf']), axis=1)  # 120% of current = 20% expansion
child_care_regulated['realistic_cap'] = child_care_regulated[['nf']].apply(
    lambda x: min(500, 0.20 * x['nf']), axis=1)  # 20% maximum

# Segment caps for realistic case (piecewise linear)
child_care_regulated['s1_max'] = 0.10 * child_care_regulated['nf']
child_care_regulated['s2_max'] = 0.05 * child_care_regulated['nf']
child_care_regulated['s3_max'] = 0.05 * child_care_regulated['nf']
child_care_regulated['overall_cap'] = child_care_regulated['realistic_cap']

# Unit costs per slot for each segment
child_care_regulated['c1'] = (20000 + 200 * child_care_regulated['nf']) / child_care_regulated['nf']
child_care_regulated['c2'] = (20000 + 400 * child_care_regulated['nf']) / child_care_regulated['nf']
child_care_regulated['c3'] = (20000 + 1000 * child_care_regulated['nf']) / child_care_regulated['nf']

# Remove facilities with nf == 0 or infinite costs
mask_valid = (
    (child_care_regulated['nf'] > 0) &
    np.isfinite(child_care_regulated['c1']) &
    np.isfinite(child_care_regulated['c2']) &
    np.isfinite(child_care_regulated['c3'])
)
child_care_regulated_valid = child_care_regulated[mask_valid].copy()

# Store facility parameters
facility_params = {}
for _, row in child_care_regulated_valid.iterrows():
    facility_params[row['facility_id']] = {
        'zipcode': row['zipcode'],
        'nf': row['nf'],
        'nf_0_5': row['nf_0_5'],
        'idealistic_cap': row['idealistic_cap'],  # For Part 1
        's1_max': row['s1_max'],
        's2_max': row['s2_max'],
        's3_max': row['s3_max'],
        'overall_cap': row['overall_cap'],  # For Part 2
        'c1': row['c1'],
        'c2': row['c2'],
        'c3': row['c3']
    }

print(f"Valid facilities for expansion: {len(facility_params)}")


Valid facilities for expansion: 14789


In [ ]:
# 10) Prepare candidate locations for Part 2
if 'site_id' not in potential_locations.columns:
    potential_locations['site_id'] = potential_locations.index.astype(str)

candidate_sites = potential_locations[['zipcode', 'latitude', 'longitude', 'site_id']].dropna()
candidate_sites = candidate_sites.rename(columns={'latitude': 'lat', 'longitude': 'lon'})

# New facility sizes and costs
sizes = {
    'S': {'tot': 100, 'max05': 50, 'cost': 65000},
    'M': {'tot': 200, 'max05': 100, 'cost': 95000},
    'L': {'tot': 400, 'max05': 200, 'cost': 115000}
}

print(f"Candidate sites: {len(candidate_sites)}")


Candidate sites: 215400


In [ ]:
# 11) Compute distance conflicts for Part 2 (Realistic)
from sklearn.neighbors import BallTree

def to_radians(df):
    return np.radians(df[['lat', 'lon']].values)

EARTH_RADIUS_MILES = 3958.8
radius_miles = 0.06
radius_radians = radius_miles / EARTH_RADIUS_MILES

new_new_conflicts = {}
new_old_conflicts = {}

# Preprocessing
existing_sites = child_care_regulated[['zipcode', 'latitude', 'longitude', 'facility_id']].dropna()
existing_sites = existing_sites.rename(columns={'latitude': 'lat', 'longitude': 'lon'})

# New-New Conflicts
for zip_code, group in candidate_sites.groupby('zipcode'):
    if len(group) < 2:
        continue
    coords = to_radians(group)
    tree = BallTree(coords, metric='haversine')
    indices = tree.query_radius(coords, r=radius_radians)
    
    pairs = set()
    for i, neighbors in enumerate(indices):
        for j in neighbors:
            if i < j:
                pairs.add((group.iloc[i]['site_id'], group.iloc[j]['site_id']))
    if pairs:
        new_new_conflicts[zip_code] = list(pairs)

# New-Old Conflicts
for zip_code, new_group in candidate_sites.groupby('zipcode'):
    old_group = existing_sites[existing_sites['zipcode'] == zip_code]
    if old_group.empty or new_group.empty:
        continue
    tree = BallTree(to_radians(old_group), metric='haversine')
    indices = tree.query_radius(to_radians(new_group), r=radius_radians)
    
    pairs = set()
    for i, neighbors in enumerate(indices):
        for j in neighbors:
            pairs.add((new_group.iloc[i]['site_id'], old_group.iloc[j]['facility_id']))
    if pairs:
        new_old_conflicts[zip_code] = list(pairs)

print(f"New-New conflicts: {sum(len(v) for v in new_new_conflicts.values())} pairs")
print(f"New-Old conflicts: {sum(len(v) for v in new_old_conflicts.values())} pairs")


New-New conflicts: 81383 pairs
New-Old conflicts: 6445 pairs


## Part 1: Idealistic Situation

In this part, we can build new facilities in any ZIP code without distance constraints. The model should:
- Allow expansion of existing facilities
- Allow building new facilities in ZIP codes
- Use simple cost structure: $20,000 fixed + $200 per slot for expansion
- $100 per 0-5 slot equipment fee


In [102]:
# Part 1: Idealistic Model 

# New facility sizes for Part 1 (same as Part 2, but simpler implementation)
sizes_ideal = {
    'S': {'tot': 100, 'max05': 50, 'cost': 65000},
    'M': {'tot': 200, 'max05': 100, 'cost': 95000},
    'L': {'tot': 400, 'max05': 200, 'cost': 115000}
}

# Get all ZIP codes from master
all_zips = master['zipcode'].unique().tolist()

# Create model with the environment that has the license activated
m_ideal = gp.Model("ideal_case", env=env)
m_ideal.setParam('OutputFlag', 1)

# --- VARIABLES ---
# Expansion per existing facility (continuous)
# For idealistic: expansion limit is min(500, 120% of current capacity)
# 120% total capacity = 20% expansion, so limit is min(500, 0.2 * nf)
x_f = {}
for f, params in facility_params.items():
    x_f[f] = m_ideal.addVar(lb=0, ub=params['idealistic_cap'], name=f"x_{f}")

# Binary indicator: 1 if facility f is expanded (for fixed cost)
z_f = {}
for f in facility_params:
    z_f[f] = m_ideal.addVar(vtype=GRB.BINARY, name=f"z_{f}")

# 0-5 slots added via expansion per facility
x_f_0_5 = {}
for f, params in facility_params.items():
    x_f_0_5[f] = m_ideal.addVar(lb=0, ub=params['idealistic_cap'], name=f"x_0_5_{f}")

# New facilities: y[zipcode][size] = number of facilities of size 'size' in ZIP
y_ideal = {}
for z in all_zips:
    y_ideal[z] = {}
    for k in sizes_ideal.keys():
        y_ideal[z][k] = m_ideal.addVar(vtype=GRB.INTEGER, lb=0, name=f"y_{z}_{k}")

# New 0-5 slots per ZIP (for new facilities)
new05_zip = {}
for z in all_zips:
    new05_zip[z] = m_ideal.addVar(lb=0, name=f"new05_{z}")

m_ideal.update()

print(f"Variables created: {len(x_f)} expansions, {len(y_ideal)} ZIPs for new facilities")


Set parameter OutputFlag to value 1
Variables created: 14789 expansions, 1646 ZIPs for new facilities


In [103]:
# --- OBJECTIVE FUNCTION ---
# Expansion cost: fixed $20,000 per facility if expanded + $200 per slot
expansion_cost = gp.quicksum(20000 * z_f[f] + 200 * x_f[f] for f in facility_params)

# Construction cost for new facilities
construction_cost = gp.quicksum(
    sizes_ideal[k]['cost'] * y_ideal[z][k] 
    for z in all_zips 
    for k in sizes_ideal.keys()
)

# Equipment cost: $100 per 0-5 slot (from expansion and new facilities)
equipment_cost = 100 * (
    gp.quicksum(x_f_0_5[f] for f in facility_params) +
    gp.quicksum(new05_zip[z] for z in all_zips)
)

m_ideal.setObjective(expansion_cost + construction_cost + equipment_cost, GRB.MINIMIZE)
print("Objective function set")


Objective function set


In [104]:
# --- CONSTRAINTS ---

# 1) Baseline cost activation: applies when expansion >= 100% of current capacity
# Problem statement: "To expand capacity by 100% or more, the state must pay a baseline cost"
# This means: if x_f >= nf (expansion is at least equal to current capacity), then z_f = 1
# Use a simple big-M constraint: x_f <= nf - epsilon + M * z_f
#   - When z_f = 0: x_f <= nf - epsilon (so x_f < nf, no baseline cost)
#   - When z_f = 1: x_f <= nf - epsilon + M (always satisfied, baseline cost applies)
# Since we minimize cost, z_f will be 0 when possible (x_f < nf), and 1 when necessary (x_f >= nf)
for f, params in facility_params.items():
    nf = params['nf']
    M = params['idealistic_cap'] + 1  # Upper bound on x_f (expansion capacity + 1)
    epsilon = 1e-6  # Small tolerance to avoid strict equality issues
    
    # If x_f >= nf, then z_f must be 1 (to allow the constraint to be satisfied)
    # This constraint forces z_f = 1 when x_f >= nf
    m_ideal.addConstr(x_f[f] <= nf - epsilon + M * z_f[f], name=f"baseline_cost_{f}")
    
    # 0-5 expansion cannot exceed total expansion
    m_ideal.addConstr(x_f_0_5[f] <= x_f[f], name=f"x_0_5_bound_{f}")

# 2) Coverage and policy constraints per ZIP
for idx, row in master.iterrows():
    zipc = row['zipcode']
    
    # Coverage threshold based on demand level
    threshold = 0.5 if row['is_high_demand'] == 1 else (1/3)
    
    # Existing totals (handle NaN values)
    existing_total_zip = row['cap_total_zip'] if pd.notna(row['cap_total_zip']) else 0
    existing_0_5_zip = row['cap_0_5_zip'] if pd.notna(row['cap_0_5_zip']) else 0
    
    # Expansion in ZIP
    expansion_total_zip = gp.quicksum(
        x_f[f] for f, p in facility_params.items() if p['zipcode'] == zipc
    )
    expansion_0_5_zip = gp.quicksum(
        x_f_0_5[f] for f, p in facility_params.items() if p['zipcode'] == zipc
    )
    
    # New facilities in ZIP
    newtotal_zip = gp.quicksum(
        sizes_ideal[k]['tot'] * y_ideal[zipc][k] 
        for k in sizes_ideal.keys()
    )
    
    # Constraint: new 0-5 slots from new facilities cannot exceed capacity
    max_new05 = gp.quicksum(
        sizes_ideal[k]['max05'] * y_ideal[zipc][k]
        for k in sizes_ideal.keys()
    )
    m_ideal.addConstr(new05_zip[zipc] <= max_new05, name=f"new05_bound_{zipc}")
    
    # Coverage constraint (0-12)
    m_ideal.addConstr(
        existing_total_zip + expansion_total_zip + newtotal_zip >= threshold * row['pop_total_0_12'],
        name=f"coverage_zip_{zipc}"
    )
    
    # Policy constraint (0-5)
    m_ideal.addConstr(
        existing_0_5_zip + expansion_0_5_zip + new05_zip[zipc] >= (2/3) * row['pop_0_5'],
        name=f"policy_0_5_zip_{zipc}"
    )

print("Constraints added")


Constraints added


In [105]:
# --- SOLVE ---
m_ideal.optimize()

# --- POSTPROCESSING ---
if m_ideal.status == GRB.OPTIMAL:
    print("\n✅ Optimization successful!")
    
    # Extract variable values
    x_vals = {f: x_f[f].X for f in facility_params}
    z_vals = {f: z_f[f].X for f in facility_params}
    x_0_5_vals = {f: x_f_0_5[f].X for f in facility_params}
    y_vals = {(z, k): y_ideal[z][k].X for z in all_zips for k in sizes_ideal.keys()}
    new05_vals = {z: new05_zip[z].X for z in all_zips}
    
    # Calculate costs
    expansion_cost_val = sum(20000 * z_vals[f] + 200 * x_vals[f] for f in facility_params)
    construction_cost_val = sum(
        sizes_ideal[k]['cost'] * y_vals[(z, k)]
        for z in all_zips
        for k in sizes_ideal.keys()
    )
    equipment_cost_val = 100 * (sum(x_0_5_vals.values()) + sum(new05_vals.values()))
    total_cost = expansion_cost_val + construction_cost_val + equipment_cost_val
    
    print(f"\nExpansion cost:      ${expansion_cost_val:,.0f}")
    print(f"Construction cost:   ${construction_cost_val:,.0f}")
    print(f"Equipment cost:      ${equipment_cost_val:,.0f}")
    print(f"Total cost:          ${total_cost:,.0f}")
    
    # Count new facilities
    new_facilities = sum(y_vals[(z, k)] for z in all_zips for k in sizes_ideal.keys())
    expanded_facilities = sum(1 for f in facility_params if z_vals[f] > 0.5)
    
    print(f"\nNew facilities built: {int(new_facilities)}")
    print(f"Facilities expanded: {expanded_facilities}")
    
    # ZIP-level summary
    zip_summary = []
    for idx, row in master.iterrows():
        z = row['zipcode']
        new_total = sum(sizes_ideal[k]['tot'] * y_vals[(z, k)] for k in sizes_ideal.keys())
        new_05 = new05_vals[z]
        exp_total = sum(x_vals[f] for f, p in facility_params.items() if p['zipcode'] == z)
        exp_05 = sum(x_0_5_vals[f] for f, p in facility_params.items() if p['zipcode'] == z)
        
        zip_summary.append({
            'zipcode': z,
            'existing_total': row['cap_total_zip'],
            'existing_0_5': row['cap_0_5_zip'],
            'expansion_total': exp_total,
            'expansion_0_5': exp_05,
            'new_total': new_total,
            'new_0_5': new_05,
            'final_total': row['cap_total_zip'] + exp_total + new_total,
            'final_0_5': row['cap_0_5_zip'] + exp_05 + new_05,
            'pop_total_0_12': row['pop_total_0_12'],
            'pop_0_5': row['pop_0_5']
        })
    
    df_ideal_results = pd.DataFrame(zip_summary)
    print("\nSample ZIP-level results:")
    print(df_ideal_results.head(10))
    
else:
    print(f"\n❌ Optimization ended with status: {m_ideal.status}")
    if m_ideal.status == GRB.INFEASIBLE:
        print("Model is infeasible. Try computing IIS to identify conflicting constraints.")
        # m_ideal.computeIIS()
        # m_ideal.write("model.ilp")


Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 24.6.0 24G90)

CPU model: Apple M3 Max
Thread count: 16 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 34516 rows, 50951 columns and 99708 nonzeros
Model fingerprint: 0x9308ffa6
Variable types: 31224 continuous, 19727 integer (14789 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+02]
  Objective range  [1e+02, 1e+05]
  Bounds range     [2e-01, 2e+02]
  RHS range        [2e-01, 1e+04]
Found heuristic solution: objective 5.094712e+08
Presolve removed 34492 rows and 50905 columns
Presolve time: 0.18s
Presolved: 24 rows, 46 columns, 92 nonzeros
Found heuristic solution: objective 4.302988e+08
Variable types: 43 continuous, 3 integer (0 binary)

Root relaxation: objective 4.301394e+08, 25 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/N

In [106]:
# ============================================================================
# PART 1 RESULTS SUMMARY - IDEALISTIC SITUATION
# ============================================================================

if m_ideal.status == GRB.OPTIMAL:
    print("=" * 80)
    print("PART 1: IDEALISTIC SITUATION - RESULTS SUMMARY")
    print("=" * 80)
    
    # Cost Breakdown (in dollars)
    print("\n📊 COST BREAKDOWN:")
    print("-" * 80)
    print(f"  Expansion Costs:        ${expansion_cost_val:>20,.2f}")
    print(f"  Construction Costs:     ${construction_cost_val:>20,.2f}")
    print(f"  Equipment Costs:        ${equipment_cost_val:>20,.2f}")
    print("-" * 80)
    print(f"  TOTAL COST:             ${total_cost:>20,.2f}")
    print("=" * 80)
    
    # Facility Statistics
    print("\n🏢 FACILITY STATISTICS:")
    print("-" * 80)
    print(f"  New facilities built:   {int(new_facilities):>20,}")
    print(f"  Facilities expanded:    {expanded_facilities:>20,}")
    
    # Count new facilities by size
    new_by_size = {}
    for z in all_zips:
        for k in sizes_ideal.keys():
            count = int(y_vals[(z, k)])
            if count > 0:
                new_by_size[k] = new_by_size.get(k, 0) + count
    
    if new_by_size:
        print("\n  New facilities by size:")
        for size in ['S', 'M', 'L']:
            count = new_by_size.get(size, 0)
            if count > 0:
                print(f"    Size {size}:                    {count:>15,}")
    
    # Capacity Statistics
    total_expansion_slots = sum(x_vals.values())
    total_expansion_0_5 = sum(x_0_5_vals.values())
    total_new_slots = sum(sizes_ideal[k]['tot'] * y_vals[(z, k)] 
                          for z in all_zips for k in sizes_ideal.keys())
    total_new_0_5 = sum(new05_vals.values())
    
    print("\n📈 CAPACITY ADDITIONS:")
    print("-" * 80)
    print(f"  Expansion slots (0-12):  {total_expansion_slots:>20,.0f}")
    print(f"  Expansion slots (0-5):  {total_expansion_0_5:>20,.0f}")
    print(f"  New facility slots (0-12): {total_new_slots:>15,.0f}")
    print(f"  New facility slots (0-5):  {total_new_0_5:>15,.0f}")
    print(f"  Total new capacity (0-12): {total_expansion_slots + total_new_slots:>13,.0f}")
    print(f"  Total new capacity (0-5):   {total_expansion_0_5 + total_new_0_5:>13,.0f}")
    
    # Coverage Statistics
    print("\n✅ COVERAGE STATISTICS:")
    print("-" * 80)
    
    # Check how many ZIPs meet coverage requirements
    zips_at_coverage_threshold = []
    zips_at_policy_threshold = []
    
    for idx, row in master.iterrows():
        z = row['zipcode']
        threshold = 0.5 if row['is_high_demand'] == 1 else (1/3)
        
        new_total = sum(sizes_ideal[k]['tot'] * y_vals[(z, k)] for k in sizes_ideal.keys())
        new_05 = new05_vals[z]
        exp_total = sum(x_vals[f] for f, p in facility_params.items() if p['zipcode'] == z)
        exp_05 = sum(x_0_5_vals[f] for f, p in facility_params.items() if p['zipcode'] == z)
        
        final_total = row['cap_total_zip'] + exp_total + new_total
        final_0_5 = row['cap_0_5_zip'] + exp_05 + new_05
        
        coverage_ratio = final_total / row['pop_total_0_12'] if row['pop_total_0_12'] > 0 else 0
        policy_ratio = final_0_5 / row['pop_0_5'] if row['pop_0_5'] > 0 else 0
        
        # Check if exactly at threshold (within small tolerance)
        if abs(coverage_ratio - threshold) < 1e-3:
            zips_at_coverage_threshold.append(z)
        if abs(policy_ratio - 2/3) < 1e-3:
            zips_at_policy_threshold.append(z)
    
    print(f"  ZIPs meeting coverage threshold: {len(zips_at_coverage_threshold):>10,}")
    print(f"  ZIPs meeting 0-5 policy threshold: {len(zips_at_policy_threshold):>8,}")
    print(f"  Total ZIPs covered:     {len(master):>20,}")
    
    print("\n" + "=" * 80)
    print("✅ PART 1 OPTIMIZATION COMPLETE")
    print("=" * 80)
else:
    print("\n❌ Cannot display results - optimization did not complete successfully.")
    print(f"   Status: {m_ideal.status}")


PART 1: IDEALISTIC SITUATION - RESULTS SUMMARY

📊 COST BREAKDOWN:
--------------------------------------------------------------------------------
  Expansion Costs:        $       12,813,250.00
  Construction Costs:     $      358,540,000.00
  Equipment Costs:        $       58,786,200.00
--------------------------------------------------------------------------------
  TOTAL COST:             $      430,139,450.00

🏢 FACILITY STATISTICS:
--------------------------------------------------------------------------------
  New facilities built:                  3,452
  Facilities expanded:                       0

  New facilities by size:
    Size S:                                658
    Size M:                                277
    Size L:                              2,517

📈 CAPACITY ADDITIONS:
--------------------------------------------------------------------------------
  Expansion slots (0-12):                64,066
  Expansion slots (0-5):                62,851
  New facility

## Part 2: Realistic Situation

In this part, we have:
- Piecewise linear expansion costs
- Discrete facility sizes at specific candidate locations
- Distance constraints (minimum 0.06 miles between facilities)
- More realistic cost structure


In [ ]:
# Part 2: Realistic Model 

# Create model with the environment that has the license activated
m = gp.Model("childcare_deserts_realistic", env=env)
m.setParam('OutputFlag', 1)

# Map candidate sites to their parameters
candidate_params = {}
site_zip_map = {}
for _, row in candidate_sites.iterrows():
    site_id = row['site_id']
    candidate_params[site_id] = {
        'zipcode': row['zipcode'],
        'lat': row['lat'],
        'lon': row['lon']
    }
    site_zip_map[site_id] = row['zipcode']

# Get all candidate site IDs
all_sites = list(candidate_params.keys())

print(f"Setting up model with {len(facility_params)} facilities and {len(all_sites)} candidate sites")


Set parameter OutputFlag to value 1
Setting up model with 14789 facilities and 215400 candidate sites


In [108]:
# --- VARIABLES ---

# Facility expansion variables (piecewise segments)
x_f1 = {}  # 0-10% expansion
x_f2 = {}  # 10-15% expansion
x_f3 = {}  # 15-20% expansion
x_f_0_5 = {}  # 0-5 portion of expansion

for f, params in facility_params.items():
    x_f1[f] = m.addVar(lb=0, ub=params['s1_max'], name=f"x_f1_{f}")
    x_f2[f] = m.addVar(lb=0, ub=params['s2_max'], name=f"x_f2_{f}")
    x_f3[f] = m.addVar(lb=0, ub=params['s3_max'], name=f"x_f3_{f}")
    x_f_0_5[f] = m.addVar(lb=0, ub=params['overall_cap'], name=f"x_f_0_5_{f}")

# New facility variables
y = {}  # y[s][k]: binary for site s, size k
new05 = {}  # new 0-5 slots at site s

for s in all_sites:
    y[s] = {}
    for k in sizes.keys():
        y[s][k] = m.addVar(vtype=GRB.BINARY, name=f"y_{s}_{k}")
    new05[s] = m.addVar(lb=0, ub=max([sizes[k]['max05'] for k in sizes]), name=f"new05_{s}")

m.update()
print("Variables created")


Variables created


In [109]:
# --- OBJECTIVE ---

# Expansion cost (piecewise linear)
expansion_cost = gp.quicksum(
    facility_params[f]['c1'] * x_f1[f] + 
    facility_params[f]['c2'] * x_f2[f] + 
    facility_params[f]['c3'] * x_f3[f] 
    for f in facility_params
)

# Construction cost for new facilities
construction_cost = gp.quicksum(
    sizes[k]['cost'] * y[s][k] 
    for s in all_sites 
    for k in sizes.keys()
)

# Equipment cost: $100 per 0-5 slot
equipment_cost = 100 * (
    gp.quicksum(x_f_0_5[f] for f in facility_params) + 
    gp.quicksum(new05[s] for s in all_sites)
)

m.setObjective(expansion_cost + construction_cost + equipment_cost, GRB.MINIMIZE)
print("Objective function set")


Objective function set


In [110]:
# --- CONSTRAINTS ---

# 1) Facility expansion constraints
for f, params in facility_params.items():
    # Total expansion cap
    m.addConstr(x_f1[f] + x_f2[f] + x_f3[f] <= params['overall_cap'], name=f"total_cap_{f}")
    # 0-5 expansion cannot exceed total expansion
    m.addConstr(x_f_0_5[f] <= x_f1[f] + x_f2[f] + x_f3[f], name=f"x_f_0_5_leq_total_{f}")

# 2) Site size selection and new05 allocation
for s in all_sites:
    # At most one size per site
    m.addConstr(gp.quicksum(y[s][k] for k in sizes.keys()) <= 1, name=f"site_size_select_{s}")
    # 0-5 slots bounded by chosen size
    m.addConstr(new05[s] <= gp.quicksum(sizes[k]['max05'] * y[s][k] for k in sizes.keys()), 
                name=f"new05_max_{s}")

print("Expansion and facility constraints added")


Expansion and facility constraints added


In [111]:
# 3) ZIP-level coverage and 0-5 policy constraints
for idx, row in master.iterrows():
    zipc = row['zipcode']
    
    # Coverage threshold: high-demand ZIPs get 0.5, else 1/3
    threshold = 0.5 if row['is_high_demand'] == 1 else (1/3)
    
    # Existing totals
    existing_total_zip = row['cap_total_zip'] if not np.isnan(row['cap_total_zip']) else 0
    existing_0_5_zip = row['cap_0_5_zip'] if not np.isnan(row['cap_0_5_zip']) else 0
    
    # Expansion in ZIP
    expansion_total_zip = gp.quicksum(
        x_f1[f] + x_f2[f] + x_f3[f] 
        for f, params in facility_params.items() 
        if params['zipcode'] == zipc
    )
    expansion_0_5_zip = gp.quicksum(
        x_f_0_5[f] 
        for f, params in facility_params.items() 
        if params['zipcode'] == zipc
    )
    
    # New facilities in ZIP
    newtotal_zip = gp.quicksum(
        gp.quicksum(sizes[k]['tot'] * y[s][k] for k in sizes.keys())
        for s in all_sites 
        if site_zip_map[s] == zipc
    )
    new05_zip = gp.quicksum(
        new05[s] 
        for s in all_sites 
        if site_zip_map[s] == zipc
    )
    
    # Coverage constraint
    m.addConstr(
        existing_total_zip + expansion_total_zip + newtotal_zip >= threshold * row['pop_total_0_12'],
        name=f"coverage_zip_{zipc}"
    )
    
    # 0-5 policy constraint
    m.addConstr(
        existing_0_5_zip + expansion_0_5_zip + new05_zip >= (2/3) * row['pop_0_5'],
        name=f"policy_0_5_zip_{zipc}"
    )

print("Coverage constraints added")


Coverage constraints added


In [ ]:
# 4) Distance conflict constraints 

# (A) New-new conflicts: for each ZIP, for each pair (s1, s2), forbid both
for zipc, pairs in new_new_conflicts.items():
    for s1, s2 in pairs:
        # Ensure both sites are in our candidate list
        if s1 in all_sites and s2 in all_sites:
            m.addConstr(
                gp.quicksum(y[s1][k] for k in y[s1]) + 
                gp.quicksum(y[s2][k] for k in y[s2]) <= 1,
                name=f"conflict_newnew_{zipc}_{s1}_{s2}"
            )

# (B) New-old conflicts: prevent building at site s if it conflicts with existing facility
# We can do this by adding a constraint that y[s][k] = 0 for all k

conflicting_sites = set()
for zipc, pairs in new_old_conflicts.items():
    for s, f in pairs:
        if s in all_sites:  # Only if site is in our candidate list
            conflicting_sites.add(s)


# for flexibility: add constraints that prevent building at sites
# that conflict with existing facilities. This is equivalent to removing them, but more explicit.
for site_id in conflicting_sites:
    if site_id in all_sites:
        # Forbid building at this site (sum of all size binaries = 0)
        m.addConstr(
            gp.quicksum(y[site_id][k] for k in sizes.keys()) == 0,
            name=f"conflict_newold_forbid_{site_id}"
        )

print(f"Distance constraints added: {len(new_new_conflicts)} ZIPs with new-new conflicts, {len(conflicting_sites)} sites with new-old conflicts")




Distance constraints added: 2154 ZIPs with new-new conflicts, 4134 sites with new-old conflicts


In [113]:
# --- SOLVE ---
m.optimize()

# --- POSTPROCESSING ---
if m.status == GRB.OPTIMAL:
    print("\n✅ Optimization successful!")
    
    # Extract variable values
    x_f1_vals = {f: x_f1[f].X for f in facility_params}
    x_f2_vals = {f: x_f2[f].X for f in facility_params}
    x_f3_vals = {f: x_f3[f].X for f in facility_params}
    x_f_0_5_vals = {f: x_f_0_5[f].X for f in facility_params}
    x_f_total = {f: x_f1_vals[f] + x_f2_vals[f] + x_f3_vals[f] for f in facility_params}
    
    y_vals = {(s, k): y[s][k].X for s in all_sites for k in sizes.keys()}
    new05_vals = {s: new05[s].X for s in all_sites}
    
    # Aggregate ZIP-level results
    added_total_zip = defaultdict(float)
    added_0_5_zip = defaultdict(float)
    
    for f, params in facility_params.items():
        z = params['zipcode']
        added_total_zip[z] += x_f_total[f]
        added_0_5_zip[z] += x_f_0_5_vals[f]
    
    for s in all_sites:
        z = site_zip_map[s]
        added_total_zip[z] += sum(sizes[k]['tot'] * y_vals[(s, k)] for k in sizes.keys())
        added_0_5_zip[z] += new05_vals[s]
    
    # Calculate costs
    expansion_cost_val = sum(
        facility_params[f]['c1'] * x_f1_vals[f] +
        facility_params[f]['c2'] * x_f2_vals[f] +
        facility_params[f]['c3'] * x_f3_vals[f]
        for f in facility_params
    )
    construction_cost_val = sum(
        sizes[k]['cost'] * y_vals[(s, k)] 
        for s in all_sites 
        for k in sizes.keys()
    )
    equipment_cost_val = 100 * (
        sum(x_f_0_5_vals.values()) + sum(new05_vals.values())
    )
    total_cost_val = expansion_cost_val + construction_cost_val + equipment_cost_val
    
    print(f"\nExpansion cost:      ${expansion_cost_val:,.0f}")
    print(f"Construction cost:   ${construction_cost_val:,.0f}")
    print(f"Equipment cost:      ${equipment_cost_val:,.0f}")
    print(f"Total cost:          ${total_cost_val:,.0f}")
    
    # Count facilities
    new_facilities_by_size = {k: sum(y_vals[(s, k)] for s in all_sites) for k in sizes.keys()}
    expanded_facilities = sum(1 for f in facility_params if x_f_total[f] > 1e-6)
    
    print(f"\nNew facilities by size:")
    for k in sizes.keys():
        print(f"  {k}: {int(new_facilities_by_size[k])}")
    print(f"Facilities expanded: {expanded_facilities}")
    
    # ZIP-level summary
    zip_summary = []
    for idx, row in master.iterrows():
        z = row['zipcode']
        zip_summary.append({
            'zipcode': z,
            'existing_total': row['cap_total_zip'],
            'existing_0_5': row['cap_0_5_zip'],
            'added_total': added_total_zip.get(z, 0),
            'added_0_5': added_0_5_zip.get(z, 0),
            'final_total': row['cap_total_zip'] + added_total_zip.get(z, 0),
            'final_0_5': row['cap_0_5_zip'] + added_0_5_zip.get(z, 0),
            'pop_total_0_12': row['pop_total_0_12'],
            'pop_0_5': row['pop_0_5'],
            'is_high_demand': row['is_high_demand']
        })
    
    df_realistic_results = pd.DataFrame(zip_summary)
    print("\nSample ZIP-level results:")
    print(df_realistic_results.head(10))
    
else:
    print(f"\n❌ Optimization ended with status: {m.status}")
    if m.status == GRB.INFEASIBLE:
        print("Model is infeasible. Try computing IIS to identify conflicting constraints.")


Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 24.6.0 24G90)

CPU model: Apple M3 Max
Thread count: 16 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 549187 rows, 920756 columns and 2825191 nonzeros
Model fingerprint: 0xd89243c9
Variable types: 274556 continuous, 646200 integer (646200 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+02]
  Objective range  [1e+02, 1e+05]
  Bounds range     [5e-02, 2e+02]
  RHS range        [2e-01, 1e+04]
Presolve removed 428826 rows and 629420 columns (presolve time = 12s)...
Presolve removed 428826 rows and 629420 columns
Presolve time: 11.66s
Presolved: 120361 rows, 291336 columns, 779193 nonzeros
Variable types: 107409 continuous, 183927 integer (182727 binary)
Deterministic concurrent LP optimizer: primal simplex, dual simplex, and barrier
Showing barrier log only...

Root barrier log...

Ordering time: 0.00s

Barrier statistics:
 AA' NZ     : 3.962e+04
 Factor NZ  : 1.687e+0

In [114]:
# ============================================================================
# PART 2 RESULTS SUMMARY - REALISTIC SITUATION
# ============================================================================

if m.status == GRB.OPTIMAL:
    print("=" * 80)
    print("PART 2: REALISTIC SITUATION - RESULTS SUMMARY")
    print("=" * 80)
    
    # Cost Breakdown (in dollars)
    print("\n📊 COST BREAKDOWN:")
    print("-" * 80)
    print(f"  Expansion Costs:        ${expansion_cost_val:>20,.2f}")
    print(f"  Construction Costs:     ${construction_cost_val:>20,.2f}")
    print(f"  Equipment Costs:        ${equipment_cost_val:>20,.2f}")
    print("-" * 80)
    print(f"  TOTAL COST:             ${total_cost_val:>20,.2f}")
    print("=" * 80)
    
    # Facility Statistics
    print("\n🏢 FACILITY STATISTICS:")
    print("-" * 80)
    print(f"  Facilities expanded:    {expanded_facilities:>20,}")
    
    total_new_facilities = sum(new_facilities_by_size.values())
    print(f"  New facilities built:   {int(total_new_facilities):>20,}")
    
    if new_facilities_by_size:
        print("\n  New facilities by size:")
        for size in ['S', 'M', 'L']:
            count = int(new_facilities_by_size.get(size, 0))
            if count > 0:
                size_label = f"Size {size} ({sizes[size]['tot']} slots)"
                print(f"    {size_label:30s} {count:>15,}")
    
    # Capacity Statistics
    total_expansion_slots = sum(x_f_total.values())
    total_expansion_0_5 = sum(x_f_0_5_vals.values())
    total_new_slots = sum(sizes[k]['tot'] * y_vals[(s, k)] 
                          for s in all_sites for k in sizes.keys())
    total_new_0_5 = sum(new05_vals.values())
    
    print("\n📈 CAPACITY ADDITIONS:")
    print("-" * 80)
    print(f"  Expansion slots (0-12):  {total_expansion_slots:>20,.0f}")
    print(f"  Expansion slots (0-5):  {total_expansion_0_5:>20,.0f}")
    print(f"  New facility slots (0-12): {total_new_slots:>15,.0f}")
    print(f"  New facility slots (0-5):  {total_new_0_5:>15,.0f}")
    print(f"  Total new capacity (0-12): {total_expansion_slots + total_new_slots:>13,.0f}")
    print(f"  Total new capacity (0-5):   {total_expansion_0_5 + total_new_0_5:>13,.0f}")
    
    # Expansion segment breakdown
    total_segment1 = sum(x_f1_vals.values())
    total_segment2 = sum(x_f2_vals.values())
    total_segment3 = sum(x_f3_vals.values())
    
    print("\n📊 EXPANSION SEGMENT BREAKDOWN:")
    print("-" * 80)
    print(f"  Segment 1 (0-10%):      {total_segment1:>20,.0f} slots")
    print(f"  Segment 2 (10-15%):     {total_segment2:>20,.0f} slots")
    print(f"  Segment 3 (15-20%):     {total_segment3:>20,.0f} slots")
    print(f"  Total expansion:         {total_expansion_slots:>20,.0f} slots")
    
    # Coverage Statistics
    print("\n✅ COVERAGE STATISTICS:")
    print("-" * 80)
    
    # Check how many ZIPs meet coverage requirements
    zips_at_coverage_threshold = []
    zips_at_policy_threshold = []
    
    for idx, row in master.iterrows():
        z = row['zipcode']
        threshold = 0.5 if row['is_high_demand'] == 1 else (1/3)
        
        final_total = row['cap_total_zip'] + added_total_zip.get(z, 0)
        final_0_5 = row['cap_0_5_zip'] + added_0_5_zip.get(z, 0)
        
        coverage_ratio = final_total / row['pop_total_0_12'] if row['pop_total_0_12'] > 0 else 0
        policy_ratio = final_0_5 / row['pop_0_5'] if row['pop_0_5'] > 0 else 0
        
        # Check if exactly at threshold (within small tolerance)
        if abs(coverage_ratio - threshold) < 1e-3:
            zips_at_coverage_threshold.append(z)
        if abs(policy_ratio - 2/3) < 1e-3:
            zips_at_policy_threshold.append(z)
    
    print(f"  ZIPs meeting coverage threshold: {len(zips_at_coverage_threshold):>10,}")
    print(f"  ZIPs meeting 0-5 policy threshold: {len(zips_at_policy_threshold):>8,}")
    print(f"  Total ZIPs covered:     {len(master):>20,}")
    
    # High-demand vs normal-demand breakdown
    high_demand_zips = master[master['is_high_demand'] == 1]
    normal_demand_zips = master[master['is_high_demand'] == 0]
    
    print("\n📋 DEMAND BREAKDOWN:")
    print("-" * 80)
    print(f"  High-demand ZIPs:       {len(high_demand_zips):>20,}")
    print(f"  Normal-demand ZIPs:     {len(normal_demand_zips):>20,}")
    
    print("\n" + "=" * 80)
    print("✅ PART 2 OPTIMIZATION COMPLETE")
    print("=" * 80)
else:
    print("\n❌ Cannot display results - optimization did not complete successfully.")
    print(f"   Status: {m.status}")


PART 2: REALISTIC SITUATION - RESULTS SUMMARY

📊 COST BREAKDOWN:
--------------------------------------------------------------------------------
  Expansion Costs:        $       16,954,412.62
  Construction Costs:     $      369,205,000.00
  Equipment Costs:        $       58,786,200.00
--------------------------------------------------------------------------------
  TOTAL COST:             $      444,945,612.62

🏢 FACILITY STATISTICS:
--------------------------------------------------------------------------------
  Facilities expanded:                   2,343
  New facilities built:                  3,513

  New facilities by size:
    Size S (100 slots)                         589
    Size M (200 slots)                         267
    Size L (400 slots)                       2,657

📈 CAPACITY ADDITIONS:
--------------------------------------------------------------------------------
  Expansion slots (0-12):                41,640
  Expansion slots (0-5):                40,569
  N